In [42]:
from langgraph.graph import StateGraph,START,END
from langchain_mistralai import ChatMistralAI
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [43]:
load_dotenv()

True

In [44]:
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0.7
)

In [45]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination : str

In [46]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}


In [47]:
def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explaination': response}


In [48]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [49]:
config = {"configurable":{"thread_id":"1"}}

In [50]:
result = workflow.invoke({"topic":"pizza"},config=config)

In [51]:
print(result)

{'topic': 'pizza', 'joke': 'Here’s a cheesy one for you:\n\n**Why did the pizza go to therapy?**\nBecause it had *too many layers* of emotional issues!\n\n*(Bonus groan-worthy follow-up: And the therapist said, "You need to *dough* better!")* 🍕😆', 'explaination': 'Here’s a breakdown of why this joke is funny (and groan-worthy):\n\n### **1. The Setup: "Why did the pizza go to therapy?"**\n- **Personification**: The joke gives the pizza human-like qualities (seeking therapy), which is absurd and unexpected. Pizzas don’t have emotions, so the idea of one needing therapy is inherently silly.\n- **Relatability**: Therapy is a common, relatable topic, but applying it to food makes it absurdly humorous. It’s like saying, *"Why did the toaster get a divorce?"*—it subverts expectations.\n\n### **2. The Punchline: "Because it had *too many layers* of emotional issues!"**\n- **Double Meaning of "Layers"**:\n  - **Literal**: Pizzas *do* have layers (crust, sauce, cheese, toppings).\n  - **Figurati

In [52]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a cheesy one for you:\n\n**Why did the pizza go to therapy?**\nBecause it had *too many layers* of emotional issues!\n\n*(Bonus groan-worthy follow-up: And the therapist said, "You need to *dough* better!")* 🍕😆', 'explaination': 'Here’s a breakdown of why this joke is funny (and groan-worthy):\n\n### **1. The Setup: "Why did the pizza go to therapy?"**\n- **Personification**: The joke gives the pizza human-like qualities (seeking therapy), which is absurd and unexpected. Pizzas don’t have emotions, so the idea of one needing therapy is inherently silly.\n- **Relatability**: Therapy is a common, relatable topic, but applying it to food makes it absurdly humorous. It’s like saying, *"Why did the toaster get a divorce?"*—it subverts expectations.\n\n### **2. The Punchline: "Because it had *too many layers* of emotional issues!"**\n- **Double Meaning of "Layers"**:\n  - **Literal**: Pizzas *do* have layers (crust, sauce, cheese, toppi

In [53]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a cheesy one for you:\n\n**Why did the pizza go to therapy?**\nBecause it had *too many layers* of emotional issues!\n\n*(Bonus groan-worthy follow-up: And the therapist said, "You need to *dough* better!")* 🍕😆', 'explaination': 'Here’s a breakdown of why this joke is funny (and groan-worthy):\n\n### **1. The Setup: "Why did the pizza go to therapy?"**\n- **Personification**: The joke gives the pizza human-like qualities (seeking therapy), which is absurd and unexpected. Pizzas don’t have emotions, so the idea of one needing therapy is inherently silly.\n- **Relatability**: Therapy is a common, relatable topic, but applying it to food makes it absurdly humorous. It’s like saying, *"Why did the toaster get a divorce?"*—it subverts expectations.\n\n### **2. The Punchline: "Because it had *too many layers* of emotional issues!"**\n- **Double Meaning of "Layers"**:\n  - **Literal**: Pizzas *do* have layers (crust, sauce, cheese, topp

In [54]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)


{'topic': 'pasta',
 'joke': 'Here’s a pasta joke for you:\n\n**Why did the spaghetti break up with the penne?**\nBecause it couldn’t *handle the twists* in their relationship!\n\n*(Bonus groan-worthy follow-up: And the linguine was just *too straight* for them.)* 🍝😆',
 'explaination': 'Here’s a breakdown of the pasta joke and why it’s funny (or at least groan-worthy):\n\n### **1. The Setup: Personifying Pasta**\nThe joke anthropomorphizes different types of pasta—spaghetti (long, thin, and straight) and penne (short, tube-shaped, with diagonal cuts that create "twists" at the ends). By framing them as a couple in a relationship, the joke plays on the idea of human emotions and breakup drama, but with food.\n\n### **2. The Punchline: "Couldn’t Handle the Twists"**\n- **Literal Meaning:** Penne pasta is literally *twisted* at the ends (the diagonal cuts create a spiral-like shape).\n- **Figurative Meaning:** In relationships, "twists" refer to unexpected complications, drama, or changes—

In [55]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Here’s a pasta joke for you:\n\n**Why did the spaghetti break up with the penne?**\nBecause it couldn’t *handle the twists* in their relationship!\n\n*(Bonus groan-worthy follow-up: And the linguine was just *too straight* for them.)* 🍝😆', 'explaination': 'Here’s a breakdown of the pasta joke and why it’s funny (or at least groan-worthy):\n\n### **1. The Setup: Personifying Pasta**\nThe joke anthropomorphizes different types of pasta—spaghetti (long, thin, and straight) and penne (short, tube-shaped, with diagonal cuts that create "twists" at the ends). By framing them as a couple in a relationship, the joke plays on the idea of human emotions and breakup drama, but with food.\n\n### **2. The Punchline: "Couldn’t Handle the Twists"**\n- **Literal Meaning:** Penne pasta is literally *twisted* at the ends (the diagonal cuts create a spiral-like shape).\n- **Figurative Meaning:** In relationships, "twists" refer to unexpected complications,

In [57]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a cheesy one for you:\n\n**Why did the pizza go to therapy?**\nBecause it had *too many layers* of emotional issues!\n\n*(Bonus groan-worthy follow-up: And the therapist said, "You need to *dough* better!")* 🍕😆', 'explaination': 'Here’s a breakdown of why this joke is funny (and groan-worthy):\n\n### **1. The Setup: "Why did the pizza go to therapy?"**\n- **Personification**: The joke gives the pizza human-like qualities (seeking therapy), which is absurd and unexpected. Pizzas don’t have emotions, so the idea of one needing therapy is inherently silly.\n- **Relatability**: Therapy is a common, relatable topic, but applying it to food makes it absurdly humorous. It’s like saying, *"Why did the toaster get a divorce?"*—it subverts expectations.\n\n### **2. The Punchline: "Because it had *too many layers* of emotional issues!"**\n- **Double Meaning of "Layers"**:\n  - **Literal**: Pizzas *do* have layers (crust, sauce, cheese, topp